# Clinic Case Study — Phase 4: Scenario Comparison

**Case study**: Community Health Clinic | **Phase**: 4 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Design a full-factorial experiment varying nurse count and exam-room count.
2. Apply Common Random Numbers (CRN) to make scenario comparisons sharper.
3. Compute 95% confidence intervals on the difference between two scenarios.
4. Make and justify a staffing recommendation based on simulation evidence.

---
> Phase 4 is about **decision support** — using simulation to answer "what if we add staff?"

In [ ]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from scipy import stats

from simdes.models.clinic import ClinicModel, ClinicParams
from simdes.analysis import confidence_interval

## Experimental Design

Factors and levels:

| Factor | Levels |
|---|---|
| `n_nurses` (triage nurses) | 1, 2, 3 |
| `n_exam_rooms` | 2, 3, 4 |

That gives **3 × 3 = 9** scenarios.  All other parameters are held at the baseline.

Response: `mean_total_time` (minutes per patient).

In [ ]:
BASE = dict(
    n_registration=1, arrival_rate=5.0/60.0,
    reg_mean=3.0, triage_mean=8.0, exam_mean=15.0, sim_time=480.0
)
N_REPS   = 30
BASE_SEED = 2024

nurses_levels = [1, 2, 3]
rooms_levels  = [2, 3, 4]

# Run all scenarios — using the SAME set of seeds for CRN
seeds = list(range(BASE_SEED, BASE_SEED + N_REPS))
rows = []

for n_nurses, n_rooms in itertools.product(nurses_levels, rooms_levels):
    p = ClinicParams(n_nurses=n_nurses, n_exam_rooms=n_rooms, **BASE)
    model = ClinicModel(params=p, seed=BASE_SEED)
    df_sc = model.run_replications(N_REPS)
    m, lo, hi = confidence_interval(df_sc['mean_total_time'].to_numpy())
    rows.append(dict(n_nurses=n_nurses, n_rooms=n_rooms,
                     mean=m, ci_lo=lo, ci_hi=hi,
                     raw=df_sc['mean_total_time'].to_numpy()))

results = pd.DataFrame(rows)
results[['n_nurses', 'n_rooms', 'mean', 'ci_lo', 'ci_hi']]

In [ ]:
# Heat map of mean total time
pivot = results.pivot(index='n_nurses', columns='n_rooms', values='mean')

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
               vmin=pivot.values.min(), vmax=pivot.values.max())
ax.set_xticks(range(len(rooms_levels)))
ax.set_xticklabels([f'{r} rooms' for r in rooms_levels])
ax.set_yticks(range(len(nurses_levels)))
ax.set_yticklabels([f'{n} nurse(s)' for n in nurses_levels])
for i, j in itertools.product(range(3), range(3)):
    ax.text(j, i, f'{pivot.values[i,j]:.1f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, label='Mean total time (min)')
ax.set_title('Clinic scenarios — mean patient time in system')
fig.tight_layout()
plt.show()

In [ ]:
# Paired-t confidence interval: baseline vs best scenario
baseline_row = results[(results.n_nurses == 1) & (results.n_rooms == 2)].iloc[0]
best_row     = results.loc[results['mean'].idxmin()]

diff = baseline_row['raw'] - best_row['raw']
t_stat, p_value = stats.ttest_rel(baseline_row['raw'], best_row['raw'])
d_mean, d_lo, d_hi = confidence_interval(diff)

print(f"Baseline  (n_nurses=1, n_rooms=2): {baseline_row['mean']:.2f} min")
print(f"Best      (n_nurses={best_row.n_nurses}, n_rooms={best_row.n_rooms}):  {best_row['mean']:.2f} min")
print(f"Reduction: {d_mean:.2f} min  95% CI [{d_lo:.2f}, {d_hi:.2f}]")
print(f"Paired t-test p-value: {p_value:.4f}")

In [ ]:
# Error bar plot: nurse count effect for fixed n_rooms=2
sub = results[results.n_rooms == 2].sort_values('n_nurses')

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(sub['n_nurses'], sub['mean'],
            yerr=[sub['mean']-sub['ci_lo'], sub['ci_hi']-sub['mean']],
            fmt='o-', capsize=5, color='tab:blue')
ax.set_xlabel('Number of triage nurses')
ax.set_ylabel('Mean total time (min)')
ax.set_title('Effect of triage nurses (n_rooms = 2 fixed)')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## Staffing Recommendation

Based on the simulation experiment:

- Adding a **second triage nurse** provides the largest reduction in patient time.
- Adding exam rooms beyond 3 yields diminishing returns.
- The recommended configuration for the 8-hour day is **2 nurses + 3 rooms**,
  which reduces mean total time by approximately X minutes compared to the baseline
  at a 95% confidence level.

*Fill in the actual value from your simulation run.*

## Try It Yourself

1. Add a factor for `n_registration` (1 or 2 clerks).  Does it interact with `n_nurses`?
2. Compute the **cost per patient** assuming $50/hr per nurse and $30/hr per exam room.
   Which scenario minimises cost-per-patient?
3. Re-run without CRN (use different seeds for each scenario).  How do the CIs compare?